In [ ]:
!pip install -q langchain langchain-google-genai


In [ ]:
import sqlite3

conn = sqlite3.connect('students.db')
cur = conn.cursor()

cur.execute('''
CREATE TABLE IF NOT EXISTS students (
student_id TEXT PRIMARY KEY,
name TEXT,
department TEXT,
python INTEGER,
database INTEGER,
ai INTEGER,
web INTEGER
)
''')

data = [
    ('22CS045', 'Dhanushya', 'Computer Science', 85, 72, 90, 78),
    ('22CS046', 'Rahul', 'Computer Science', 65, 70, 68, 72),
    ('22CS047', 'Priya', 'Information Technology', 92, 88, 95, 90),
    ('22CS048', 'Arun', 'Information Technology', 55, 60, 58, 62),
    ('22CS049', 'Meena', 'Computer Science', 78, 85, 80, 88)
]

cur.executemany('''
INSERT OR REPLACE INTO students
(student_id, name, department, python, database, ai, web)
VALUES (?, ?, ?, ?, ?, ?, ?)
''', data)

conn.commit()
conn.close()


In [ ]:
from langchain.tools import tool
import sqlite3

def db():
    return sqlite3.connect('students.db')


In [ ]:
@tool
def get_student_info(student_id: str) -> str:
    """Get a student's name and department."""
    conn = db()
    cur = conn.cursor()
    cur.execute('SELECT name, department FROM students WHERE student_id = ?', (student_id,))
    row = cur.fetchone()
    conn.close()
    if not row:
        return 'Student not found'
    return f'Name: {row[0]}, Department: {row[1]}'


In [ ]:
@tool
def get_student_marks(student_id: str) -> str:
    """Get all subject marks for a student."""
    conn = db()
    cur = conn.cursor()
    cur.execute('SELECT python, database, ai, web FROM students WHERE student_id = ?', (student_id,))
    row = cur.fetchone()
    conn.close()
    if not row:
        return 'Student not found'
    return f'Python: {row[0]}, Database: {row[1]}, AI: {row[2]}, Web: {row[3]}'


In [ ]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""
    try:
        return str(eval(expression, {'__builtins__': {}}, {}))
    except Exception as e:
        return f'Error: {e}'


In [ ]:
@tool
def get_passing_rules() -> str:
    """Get the university passing requirements."""
    return 'Minimum overall average: 40%. Minimum mark in each subject: 35%.'


In [ ]:
tools = [
    get_student_info,
    get_student_marks,
    calculator,
    get_passing_rules
]


In [ ]:
import os
from getpass import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ['GOOGLE_API_KEY'] = getpass('Enter Gemini API key: ')

llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0
)


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt='''You are a student information assistant. Use the tools when needed. Use get_student_info for name and department, get_student_marks for marks, calculator for totals and averages, and get_passing_rules for passing requirements. Do not guess student information. Decide yourself which tools are needed and whether another tool is required after receiving a result.'''
)


In [ ]:
while True:
    question = input('You: ')
    if question.lower() == 'exit':
        break
    result = agent.invoke({'messages': [{'role': 'user', 'content': question}]})
    print('\nAgent:', result['messages'][-1].content)
    print()
